# Langchain Docs, References: 

[Langchain Docs](https://docs.langchain.com/)

[Langchain Python Reference](https://reference.langchain.com/python/)

![alt text](<../../assets/Full Rag Pipeline.png>)

### Environment Initialization 

In [ ]:
import warnings
import os 
from dotenv import load_dotenv

# 0. Disable Warnings
warnings.filterwarnings("ignore")

# 1. Add parentheses to actually run the function
load_dotenv()

try: 
    # 2. Use .get() with a default empty string "" to avoid NoneType errors
    os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGSMITH_TRACING_V2")
    os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
    os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGSMITH_PROJECT")
    os.environ["LANGCHAIN_ENDPOINT"] = os.getenv("LANGSMITH_ENDPOINT")
    os.environ["MISTRAL_API_KEY"] = os.getenv("MISTRAL_API_KEY")
    os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
    os.environ["USER_AGENT"] = "MyLangChainApp/1.0" # For WebBaseLoader
    print("Environment variables set successfully")
except Exception as e: 
    print(f"Error: {e}")

Environment variables set successfully


### Part 1: Overview

In [ ]:
import bs4
# from langchain.text_splitter import RecursiveCharacterTextSplitter # Old import
from langchain_text_splitters import RecursiveCharacterTextSplitter # New import
from langchain_community.document_loaders import WebBaseLoader # New import
from langchain_community.vectorstores import Chroma # New import
from langchain_core.output_parsers import StrOutputParser # New import
from langchain_core.runnables import RunnablePassthrough # New import
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings # New import
from langsmith import Client
client = Client()


### Indexing ### 

# Load Documents 
loader = WebBaseLoader(
    web_paths = ["https://lilianweng.github.io/posts/2023-06-23-agent/"],
    bs_kwargs = dict(
        parse_only = bs4.SoupStrainer(
            class_ = ("post-content", "post-title", "author")
        )
    ),
)

docs = loader.load()

# Split
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200,
)
splits = text_splitter.split_documents(docs)


# Embed and Store
vectorstore = Chroma.from_documents(
    documents = splits,
    embedding = MistralAIEmbeddings(model = "mistral-embed"),
    collection_name = "Tutorial", 
    persist_directory = "../db"
)

# Retrieve
retriever = vectorstore.as_retriever()

prompt = client.pull_prompt("rlm/rag-prompt")

# LLM
llm = ChatMistralAI(
    model = "mistral-medium-latest",
    temperature = 0,
)

# Post Processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# RAG Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
    )

# Question 

rag_chain.invoke("what is Task Decomposition?")

'Task decomposition is the process of breaking down a complex task into smaller, manageable subgoals or steps. This can be done using simple prompts (e.g., "Steps for XYZ"), task-specific instructions, or human input. Another approach, like **LLM+P**, outsources planning to an external tool using structured languages like PDDL.'

### Part 2: Indexing

In [7]:
question = "What kinds of pets do I like?"
document =  "My favorite pet is cat."

In [8]:
import tiktoken

def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Return the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

num_of_tokens = num_tokens_from_string(question, "cl100k_base")

num_of_tokens

8

In [9]:
embed = MistralAIEmbeddings(model = "mistral-embed")
query_result = embed.embed_query(question)
doc_result = embed.embed_query(document)

len(query_result)

1024

In [10]:
import numpy as np 

def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product / (norm_vec1 * norm_vec2) 

similarity_result = cosine_similarity(query_result, doc_result)

print(similarity_result) # Very Similar 

0.7558983474652954


In [11]:
### Indexing 

# Load Documents 
loader = WebBaseLoader(
    web_paths = ["https://lilianweng.github.io/posts/2023-06-23-agent/"],
    bs_kwargs = dict(
        parse_only = bs4.SoupStrainer(
            class_ = ("post-content", "post-title", "author")
        )
    ),
)

blog_docs = loader.load()

In [12]:
# Split
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size = 300,
    chunk_overlap = 50,
)

# Make Splits
splits = text_splitter.split_documents(blog_docs)


In [13]:
# Vector Store
embeddings = MistralAIEmbeddings(model = "mistral-embed")

vStore = Chroma.from_documents(
    splits,
    embedding=embeddings,
    collection_name="blog_posts",
    persist_directory="../db_blog",
)

retriever = vStore.as_retriever()


### Part 3:  Retrieval 

In [ ]:
embeddings = MistralAIEmbeddings(model = "mistral-embed")
# To load the vector store we previously created
vStore = Chroma(
    persist_directory="../db_blog",
    embedding_function=embeddings,
    collection_name="blog_posts",
)

retriever = vStore.as_retriever(search_kwargs={"k": 1})

docs = retriever.invoke("what is Task Decomposition?")
len(docs)

1

### Part 4: Generation

In [ ]:
from langchain_mistralai import ChatMistralAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
# Prompt Template 
template = """Answer the question based only on the following context: 
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context: \n{context}\n\nQuestion: {question}\n'), additional_kwargs={})])

In [31]:
# LLM
llm = ChatMistralAI(model = "mistral-small-latest", temperature = 0)

# Chain
chain = prompt | llm | StrOutputParser()

# Run
result = chain.invoke({"context": blog_docs, "question": "What is Task Decomposition?"})

print(result)

Task decomposition is the process of breaking down complex tasks into smaller, manageable subgoals or steps. It is often achieved through techniques like Chain of Thought (CoT) or Tree of Thoughts (ToT), which help decompose tasks into simpler, sequential steps. This approach enhances problem-solving by making large tasks more manageable and interpretable.


In [32]:
from langsmith import Client
client = Client()
prompt_temp = client.pull_prompt("rlm/rag-prompt")

# RAG Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()} 
    | prompt_temp
    | llm
    | StrOutputParser()
)

# Test
print(rag_chain.invoke("What is Task Decomposition?"))

Task decomposition is the process of breaking down complex tasks into smaller, more manageable steps. It helps agents plan and execute tasks efficiently by simplifying the problem-solving process. This can be done through prompting, task-specific instructions, or human input.
